# Whole Foods Location Similarity Model

## Project Objective

The objective of this project is to identify census tracts that resemble neighborhoods where Whole Foods currently operates.

Using tract-level demographic, crime, and retail environment data, we construct a similarity-based scoring model to rank candidate locations based on their similarity to existing Whole Foods neighborhoods.

## Analytical Approach

The analysis proceeds in four stages:

1. Build a master tract-level dataset by merging demographic, crime, retail, and store presence data.
2. Select and standardize key neighborhood characteristics.
3. Estimate the typical profile of existing Whole Foods locations.
4. Rank candidate tracts by similarity to this profile.

## Output

The final output includes:

- a tract-level analytical dataset for dashboard analysis
- a ranked table of candidate tracts that most closely resemble current Whole Foods neighborhoods

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Environment Setup and Data Inputs

This section initializes the Spark session, defines project paths, and loads the tract-level datasets generated during the preprocessing stage.

### Data Inputs
The analysis uses four tract-level input tables:

- `tract_acs.csv`
- `tract_crime.csv`
- `tract_osm.csv`
- `tract_wf.csv`

These tables are treated as processed inputs to the modeling pipeline.

In [2]:
# 1.1 Initialize Spark and define project paths

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Create Spark session
spark = SparkSession.builder \
    .appName("WholeFoodsLocationAnalysis") \
    .getOrCreate()

# Define the base project path in Google Drive
BASE_PATH = "/content/drive/MyDrive/msba405-group-project/"
PROCESSED_PATH = BASE_PATH + "processed_data/"

# Define full file paths
ACS_PATH = PROCESSED_PATH + "tract_acs.csv"
CRIME_PATH = PROCESSED_PATH + "tract_crime.csv"
OSM_PATH = PROCESSED_PATH + "tract_osm.csv"
WF_PATH = PROCESSED_PATH + "tract_wf.csv"

In [3]:
# 1.2 Load processed tract-level input tables

# Read ACS tract-level data
acs = spark.read.csv(
    ACS_PATH,
    header=True,
    inferSchema=True
)

# Read crime tract-level data
crime = spark.read.csv(
    CRIME_PATH,
    header=True,
    inferSchema=True
)

# Read OSM tract-level data
osm = spark.read.csv(
    OSM_PATH,
    header=True,
    inferSchema=True
)

# Read Whole Foods tract-level data
wf = spark.read.csv(
    WF_PATH,
    header=True,
    inferSchema=True
)

## 2. Master Tract Table Construction

Before modeling, the four tract-level tables must be aligned into a single analytical dataset.

This stage performs the following data engineering checks:

- verify schema consistency across inputs
- standardize `GEOID` as the common tract identifier
- confirm one row per tract in each source table
- merge all sources into a unified master table using ACS as the backbone dataset

ACS is used as the base table because it provides the broadest tract coverage and contains the core demographic variables used later in the scoring model.

In [4]:
# 2.1 Inspect schemas and join keys

# Print row counts
print("ACS row count:", acs.count())
print("Crime row count:", crime.count())
print("OSM row count:", osm.count())
print("WF row count:", wf.count())

# Print column names
print("ACS columns:", acs.columns)
print("Crime columns:", crime.columns)
print("OSM columns:", osm.columns)
print("WF columns:", wf.columns)

# Print schemas
print("ACS schema:")
acs.printSchema()

print("Crime schema:")
crime.printSchema()

print("OSM schema:")
osm.printSchema()

print("WF schema:")
wf.printSchema()

# Preview the GEOID column in each dataset
acs.select("GEOID").show(5, truncate=False)
crime.select("GEOID").show(5, truncate=False)
osm.select("GEOID").show(5, truncate=False)
wf.select("GEOID").show(5, truncate=False)

ACS row count: 2498
Crime row count: 2463
OSM row count: 2498
WF row count: 2498
ACS columns: ['GEOID', 'income', 'ba_rate', 'pop', 'pop_density']
Crime columns: ['GEOID', 'violent_per_1000', 'property_per_1000', 'crime_low_60']
OSM columns: ['GEOID', 'retail_density']
WF columns: ['GEOID', 'has_wf']
ACS schema:
root
 |-- GEOID: long (nullable = true)
 |-- income: double (nullable = true)
 |-- ba_rate: double (nullable = true)
 |-- pop: double (nullable = true)
 |-- pop_density: double (nullable = true)

Crime schema:
root
 |-- GEOID: long (nullable = true)
 |-- violent_per_1000: double (nullable = true)
 |-- property_per_1000: double (nullable = true)
 |-- crime_low_60: boolean (nullable = true)

OSM schema:
root
 |-- GEOID: long (nullable = true)
 |-- retail_density: double (nullable = true)

WF schema:
root
 |-- GEOID: long (nullable = true)
 |-- has_wf: integer (nullable = true)

+----------+
|GEOID     |
+----------+
|6037101110|
|6037101122|
|6037101220|
|6037101221|
|6037101222|

In [5]:
# 2.2 Standardize GEOID across source tables

# Cast GEOID to string in every dataset
acs = acs.withColumn("GEOID", col("GEOID").cast("string"))
crime = crime.withColumn("GEOID", col("GEOID").cast("string"))
osm = osm.withColumn("GEOID", col("GEOID").cast("string"))
wf = wf.withColumn("GEOID", col("GEOID").cast("string"))

In [6]:
# 2.3 Check for duplicate tract identifiers

from pyspark.sql.functions import count

# Check duplicate GEOIDs in ACS
acs.groupBy("GEOID").count().filter(col("count") > 1).show()

# Check duplicate GEOIDs in crime
crime.groupBy("GEOID").count().filter(col("count") > 1).show()

# Check duplicate GEOIDs in OSM
osm.groupBy("GEOID").count().filter(col("count") > 1).show()

# Check duplicate GEOIDs in WF
wf.groupBy("GEOID").count().filter(col("count") > 1).show()

+-----+-----+
|GEOID|count|
+-----+-----+
+-----+-----+

+-----+-----+
|GEOID|count|
+-----+-----+
+-----+-----+

+-----+-----+
|GEOID|count|
+-----+-----+
+-----+-----+

+-----+-----+
|GEOID|count|
+-----+-----+
+-----+-----+



In [7]:
# 2.4 Merge tract-level sources into a master table

# Start with ACS as the base table
master_df = acs

# Join crime dataset
master_df = master_df.join(
    crime,
    on="GEOID",
    how="left"
)

# Join OSM dataset
master_df = master_df.join(
    osm,
    on="GEOID",
    how="left"
)

# Join Whole Foods dataset
master_df = master_df.join(
    wf,
    on="GEOID",
    how="left"
)

In [8]:
# 2.5 Validate the merged dataset

# Check row count
print("Master table row count:", master_df.count())

# Print columns
print("\nMaster table columns:")
print(master_df.columns)

# Print schema
print("\nMaster table schema:")
master_df.printSchema()

# Preview rows
master_df.show(5, truncate=False)

# Check duplicate GEOIDs after merge
master_df.groupBy("GEOID").count().filter(col("count") > 1).show()

Master table row count: 2498

Master table columns:
['GEOID', 'income', 'ba_rate', 'pop', 'pop_density', 'violent_per_1000', 'property_per_1000', 'crime_low_60', 'retail_density', 'has_wf']

Master table schema:
root
 |-- GEOID: string (nullable = true)
 |-- income: double (nullable = true)
 |-- ba_rate: double (nullable = true)
 |-- pop: double (nullable = true)
 |-- pop_density: double (nullable = true)
 |-- violent_per_1000: double (nullable = true)
 |-- property_per_1000: double (nullable = true)
 |-- crime_low_60: boolean (nullable = true)
 |-- retail_density: double (nullable = true)
 |-- has_wf: integer (nullable = true)

+----------+------------------+-------------------+------------------+------------------+------------------+------------------+------------+--------------+------+
|GEOID     |income            |ba_rate            |pop               |pop_density       |violent_per_1000  |property_per_1000 |crime_low_60|retail_density|has_wf|
+----------+------------------+------

## 3. Feature Selection and Candidate Definition

This section extracts the variables used in the similarity model and defines the candidate tract universe.

The model focuses on neighborhood characteristics that are plausibly associated with existing Whole Foods locations:

- household income
- bachelor degree attainment
- population density
- retail density

Two operational filters are then applied:

- exclude tracts with unfavorable crime conditions
- exclude tracts that already contain an existing Whole Foods store

In [9]:
# 3.1 Select modeling variables

features_df = master_df.select(
    "GEOID",
    "income",
    "ba_rate",
    "pop_density",
    "retail_density",
    "crime_low_60",
    "has_wf"
)

features_df.show(5)

+----------+------------------+-------------------+------------------+--------------+------------+------+
|     GEOID|            income|            ba_rate|       pop_density|retail_density|crime_low_60|has_wf|
+----------+------------------+-------------------+------------------+--------------+------------+------+
|6037101110| 80500.33333333333| 0.2838001954867394| 3635.617732594188|           0.0|       false|     0|
|6037101122|          107695.0| 0.3672074949333901| 1574.103315185634|           0.0|        true|     0|
|6037101220| 75710.66666666667|0.32366736251934797| 4951.689784409284|           0.0|       false|     0|
|6037101221|53116.666666666664|0.26242308339520704|10616.130872047388|           0.0|       false|     0|
|6037101222|40188.666666666664| 0.1592546043761676| 8622.472839772645|           0.0|       false|     0|
+----------+------------------+-------------------+------------------+--------------+------------+------+
only showing top 5 rows


In [10]:
# 3.2 Define candidate tracts

candidate_df = features_df.filter(
    (col("crime_low_60") == True) &
    (col("has_wf") == 0)
)

# Remove rows with missing feature values
candidate_df = candidate_df.dropna(
    subset=["income", "ba_rate", "pop_density", "retail_density"]
)

print("Candidate tract count:", candidate_df.count())

candidate_df.show(5)

Candidate tract count: 1358
+----------+------------------+-------------------+------------------+--------------+------------+------+
|     GEOID|            income|            ba_rate|       pop_density|retail_density|crime_low_60|has_wf|
+----------+------------------+-------------------+------------------+--------------+------------+------+
|6037101122|          107695.0| 0.3672074949333901| 1574.103315185634|           0.0|        true|     0|
|6037102104|107788.66666666667| 0.4404588908636327| 2474.847125542227|           0.0|        true|     0|
|6037106111|116512.33333333333| 0.2426906834445083| 1740.536120209696|           0.0|        true|     0|
|6037106408|55622.333333333336|0.08882292516575281|16540.795390055442|           0.0|        true|     0|
|6037108203|143055.66666666666| 0.6206087034390753|2283.3811820827077|           0.0|        true|     0|
+----------+------------------+-------------------+------------------+--------------+------------+------+
only showing top 5

## 4. Feature Standardization and Reference Profile

Because the selected variables are measured on different scales, all modeling features are converted to z-scores before similarity is computed.

After standardization, the notebook computes the centroid of existing Whole Foods tracts in standardized feature space. This centroid represents the reference neighborhood profile against which candidate tracts will be compared.

In [11]:
# 4.1 Compute standardization statistics

from pyspark.sql.functions import mean, stddev

stats = features_df.select(
    mean("income").alias("income_mean"),
    stddev("income").alias("income_std"),
    mean("ba_rate").alias("ba_mean"),
    stddev("ba_rate").alias("ba_std"),
    mean("pop_density").alias("pop_mean"),
    stddev("pop_density").alias("pop_std"),
    mean("retail_density").alias("retail_mean"),
    stddev("retail_density").alias("retail_std")
).collect()[0]

In [12]:
# 4.2 Transform features into standardized space

from pyspark.sql.functions import col

features_z_df = features_df.withColumn(
    "z_income",
    (col("income") - stats["income_mean"]) / stats["income_std"]
).withColumn(
    "z_ba_rate",
    (col("ba_rate") - stats["ba_mean"]) / stats["ba_std"]
).withColumn(
    "z_pop_density",
    (col("pop_density") - stats["pop_mean"]) / stats["pop_std"]
).withColumn(
    "z_retail_density",
    (col("retail_density") - stats["retail_mean"]) / stats["retail_std"]
)

# Rebuild candidate set in standardized space
candidate_z_df = features_z_df.filter(
    (col("crime_low_60") == True) &
    (col("has_wf") == 0)
).dropna(
    subset=["income", "ba_rate", "pop_density", "retail_density"]
)

candidate_z_df.show(5)

+----------+------------------+-------------------+------------------+--------------+------------+------+-------------------+--------------------+-------------------+-------------------+
|     GEOID|            income|            ba_rate|       pop_density|retail_density|crime_low_60|has_wf|           z_income|           z_ba_rate|      z_pop_density|   z_retail_density|
+----------+------------------+-------------------+------------------+--------------+------------+------+-------------------+--------------------+-------------------+-------------------+
|6037101122|          107695.0| 0.3672074949333901| 1574.103315185634|           0.0|        true|     0|0.43116547600476446| 0.08363017295498885|-0.8203205629123572|-0.4527389043218039|
|6037102104|107788.66666666667| 0.4404588908636327| 2474.847125542227|           0.0|        true|     0|0.43368181945061746| 0.42165989915130797|-0.6152008988859959|-0.4527389043218039|
|6037106111|116512.33333333333| 0.2426906834445083| 1740.53612020

In [13]:
# 4.3 Compute the Whole Foods centroid

wf_z_profile = features_z_df.filter(
    col("has_wf") == 1
).select(
    "z_income",
    "z_ba_rate",
    "z_pop_density",
    "z_retail_density"
).groupBy().avg()

wf_z_stats = wf_z_profile.collect()[0]

wf_z_income = wf_z_stats["avg(z_income)"]
wf_z_ba = wf_z_stats["avg(z_ba_rate)"]
wf_z_pop = wf_z_stats["avg(z_pop_density)"]
wf_z_retail = wf_z_stats["avg(z_retail_density)"]

wf_z_profile.show()

+------------------+------------------+-------------------+---------------------+
|     avg(z_income)|    avg(z_ba_rate)| avg(z_pop_density)|avg(z_retail_density)|
+------------------+------------------+-------------------+---------------------+
|0.8268627216048501|1.2909161190004192|-0.2039504952263979|   1.0386191571601666|
+------------------+------------------+-------------------+---------------------+



## 5. Similarity Scoring and Ranking

Similarity is measured as the inverse transformation of Euclidean distance between each candidate tract and the standardized Whole Foods centroid.

A smaller distance indicates that a tract more closely resembles the observed profile of existing Whole Foods neighborhoods. For interpretability, distance is converted into a bounded similarity score:

$$
\textbf{score}_i = \frac{1}{1 + d_i}
$$

Higher values indicate stronger similarity.

In [14]:
# 5.1 Compute Euclidean distance

from pyspark.sql.functions import sqrt

scored_df = candidate_z_df.withColumn(
    "distance",
    sqrt(
        (col("z_income") - wf_z_income) ** 2 +
        (col("z_ba_rate") - wf_z_ba) ** 2 +
        (col("z_pop_density") - wf_z_pop) ** 2 +
        (col("z_retail_density") - wf_z_retail) ** 2
    )
)

scored_df.select(
    "GEOID",
    "z_income",
    "z_ba_rate",
    "z_pop_density",
    "z_retail_density",
    "distance"
).show(5)

+----------+-------------------+--------------------+-------------------+-------------------+------------------+
|     GEOID|           z_income|           z_ba_rate|      z_pop_density|   z_retail_density|          distance|
+----------+-------------------+--------------------+-------------------+-------------------+------------------+
|6037101122|0.43116547600476446| 0.08363017295498885|-0.8203205629123572|-0.4527389043218039|2.0538200002874545|
|6037102104|0.43368181945061746| 0.42165989915130797|-0.6152008988859959|-0.4527389043218039| 1.817545971804462|
|6037106111| 0.6680420483524524|-0.49097162350098755|-0.7824200626180666|-0.4527389043218039| 2.399817459948708|
|6037106408|-0.9677602906295594| -1.2010178354658778|  2.587932499373956|-0.4527389043218039| 4.410120872782201|
|6037108203|  1.381125424129195|  1.2529886546768187|-0.6588020089278834|-0.4527389043218039| 1.655199203426404|
+----------+-------------------+--------------------+-------------------+-------------------+---

In [15]:
# 5.2 Convert distance to similarity score

scored_df = scored_df.withColumn(
    "score",
    1 / (1 + col("distance"))
)

In [16]:
# 5.3 Rank candidate locations

ranked_df = scored_df.orderBy(col("score").desc())

ranked_df.select(
    "GEOID",
    "income",
    "ba_rate",
    "pop_density",
    "retail_density",
    "distance",
    "score"
).show(10)

+----------+------------------+------------------+------------------+------------------+-------------------+------------------+
|     GEOID|            income|           ba_rate|       pop_density|    retail_density|           distance|             score|
+----------+------------------+------------------+------------------+------------------+-------------------+------------------+
|6037620601|          123234.0|0.5723884600780763| 4365.828452987836|  20.5025767345546| 0.2630188519996123|0.7917538193644531|
|6037480704|114215.66666666667|0.6515311339551277|3824.8540393182902|16.831726682681744|0.36364510906416014|0.7333286302667694|
|6037702201|          110573.0|0.6688524190977461|3329.3913251459153|22.422939830117166|0.45753970941026395| 0.686087654108999|
|6037700802|          116588.0| 0.598649042416282| 5924.084174524103|17.509608397608186|0.47273704931009825|0.6790078381395027|
|6037310300|136226.66666666666|0.5970015804784005|2860.6169345883986|21.673720049039147| 0.5248662714145

For visualization and dashboard presentation, numeric values are rounded
to improve readability. The rounding only affects display values and does
not change the underlying analytical results.

In [17]:
from pyspark.sql.functions import round

dashboard_df = ranked_df.select(
    "GEOID",
    round("income",0).alias("income"),
    round("ba_rate",3).alias("ba_rate"),
    round("pop_density",1).alias("pop_density"),
    round("retail_density",2).alias("retail_density"),
    round("distance",3).alias("distance"),
    round("score",3).alias("score")
)

dashboard_df.show(10)

+----------+--------+-------+-----------+--------------+--------+-----+
|     GEOID|  income|ba_rate|pop_density|retail_density|distance|score|
+----------+--------+-------+-----------+--------------+--------+-----+
|6037620601|123234.0|  0.572|     4365.8|          20.5|   0.263|0.792|
|6037480704|114216.0|  0.652|     3824.9|         16.83|   0.364|0.733|
|6037702201|110573.0|  0.669|     3329.4|         22.42|   0.458|0.686|
|6037700802|116588.0|  0.599|     5924.1|         17.51|   0.473|0.679|
|6037310300|136227.0|  0.597|     2860.6|         21.67|   0.525|0.656|
|6037621301|117529.0|   0.64|     4575.9|          12.8|   0.569|0.637|
|6037700901|105686.0|  0.696|     3960.8|         18.03|   0.573|0.636|
|6037621324|114294.0|  0.719|     5615.9|          17.2|   0.602|0.624|
|6037577200|108071.0|   0.62|     5794.2|         14.24|   0.681|0.595|
|6037554519|121399.0|  0.562|     1995.2|         15.82|   0.686|0.593|
+----------+--------+-------+-----------+--------------+--------

## 6. Model Validation

To assess whether the similarity model captures meaningful neighborhood characteristics, we test whether tracts with existing Whole Foods stores receive systematically higher similarity scores than other tracts.

The validation focuses on three checks:

1. average similarity score for Whole Foods vs. non-Whole Foods tracts  
2. median similarity score comparison  
3. prevalence of existing Whole Foods tracts among high-scoring locations

In [18]:
# 6.1 Compare score distributions

from pyspark.sql.functions import col, sqrt, avg, round, expr

# Score all tracts in standardized feature space
all_scored_df = features_z_df.withColumn(
    "distance",
    sqrt(
        (col("z_income") - wf_z_income) ** 2 +
        (col("z_ba_rate") - wf_z_ba) ** 2 +
        (col("z_pop_density") - wf_z_pop) ** 2 +
        (col("z_retail_density") - wf_z_retail) ** 2
    )
)

all_scored_df = all_scored_df.withColumn(
    "score",
    1 / (1 + col("distance"))
)

In [19]:
# Average score comparison

print("Validation A1: Average similarity score by Whole Foods presence")

avg_validation_df = all_scored_df.groupBy("has_wf").agg(
    round(avg("score"), 3).alias("avg_score")
)

avg_validation_df.show()

Validation A1: Average similarity score by Whole Foods presence
+------+---------+
|has_wf|avg_score|
+------+---------+
|     1|    0.401|
|     0|    0.304|
+------+---------+



In [20]:
# Median score comparison

print("Validation A2: Median and average similarity score by Whole Foods presence")

median_validation_df = all_scored_df.groupBy("has_wf").agg(
    round(expr("percentile_approx(score, 0.5)"), 3).alias("median_score"),
    round(avg("score"), 3).alias("avg_score")
)

median_validation_df.show()


Validation A2: Median and average similarity score by Whole Foods presence
+------+------------+---------+
|has_wf|median_score|avg_score|
+------+------------+---------+
|     1|       0.402|    0.401|
|     0|       0.286|    0.304|
+------+------------+---------+



In [21]:
# 6.2 Evaluate recovery of existing Whole Foods tracts

top_n_all = 50

print(f"Validation A3: Existing Whole Foods tracts in Top {top_n_all} of all tracts")

topn_all_df = all_scored_df.orderBy(col("score").desc()).limit(top_n_all)

topn_all_df.select(
    "GEOID",
    "has_wf",
    round("score", 3).alias("score"),
    round("distance", 3).alias("distance")
).show(20, truncate=False)

wf_in_topn_all = topn_all_df.filter(col("has_wf") == 1).count()
total_wf_tracts = all_scored_df.filter(col("has_wf") == 1).count()

recovery_rate = wf_in_topn_all / total_wf_tracts
share_top_n = wf_in_topn_all / top_n_all

print(f"Existing Whole Foods tracts in Top {top_n_all}: {wf_in_topn_all}")
print("Total existing Whole Foods tracts:", total_wf_tracts)
print("Recovery rate:", __builtins__.round(recovery_rate, 3))
print("Share of Top N:", __builtins__.round(share_top_n, 3))

Validation A3: Existing Whole Foods tracts in Top 50 of all tracts
+----------+------+-----+--------+
|GEOID     |has_wf|score|distance|
+----------+------+-----+--------+
|6037214700|0     |0.829|0.207   |
|6037620601|0     |0.792|0.263   |
|6037272100|0     |0.765|0.307   |
|6037480704|0     |0.733|0.364   |
|6037271200|0     |0.704|0.42    |
|6037702201|0     |0.686|0.458   |
|6037700802|0     |0.679|0.473   |
|6037272301|0     |0.679|0.473   |
|6037269700|0     |0.659|0.518   |
|6037310300|0     |0.656|0.525   |
|6037188300|0     |0.641|0.56    |
|6037621301|0     |0.637|0.569   |
|6037700901|0     |0.636|0.573   |
|6037621324|0     |0.624|0.602   |
|6037214000|0     |0.61 |0.639   |
|6037463602|1     |0.609|0.642   |
|6037271703|0     |0.596|0.678   |
|6037577200|0     |0.595|0.681   |
|6037267700|0     |0.593|0.686   |
|6037554519|0     |0.593|0.686   |
+----------+------+-----+--------+
only showing top 20 rows
Existing Whole Foods tracts in Top 50: 2
Total existing Whole Foods 

### Validation Results

The validation results suggest that the similarity-based model captures
meaningful characteristics of Whole Foods locations. Tracts that already
contain Whole Foods stores have higher similarity scores, with a median score
of **0.402** compared to **0.286** for other tracts (average: **0.401 vs 0.304**).

Among the top 50 scoring tracts in the dataset, **2 already contain Whole Foods
stores** out of **28 existing Whole Foods tracts**, corresponding to a recovery
rate of **0.071**.

Although the recovery rate is modest, this outcome is expected because the model relies on a limited set of tract-level indicators. Additional features such as consumer spending patterns, real estate variables, or competitive retail presence could further improve the model's ability to recover existing Whole Foods locations.

Overall, the validation results suggest that the similarity model captures meaningful aspects of Whole Foods neighborhood profiles and provides a reasonable starting point for identifying potential expansion locations.

## 7. Export datasets for the Snowflake serving layer

To support downstream analytics and dashboard applications, we export the final processed datasets for loading into Snowflake.

Two datasets are produced:

1. a tract-level analytical dataset for exploratory analysis and visualization
2. a ranked candidate location table for potential Whole Foods expansion analysis

In [22]:
# Export final datasets for Snowflake

from pyspark.sql.functions import round, col

# 1. Full tract-level serving table
serving_df = all_scored_df.select(
    "GEOID",
    "income",
    "ba_rate",
    "pop_density",
    "retail_density",
    "crime_low_60",
    "has_wf",
    "distance",
    "score"
)

# 2. Top candidate table for dashboard / presentation
top_candidates_export_df = ranked_df.limit(10).select(
    "GEOID",
    round("income", 0).alias("income"),
    round("ba_rate", 3).alias("ba_rate"),
    round("pop_density", 1).alias("pop_density"),
    round("retail_density", 2).alias("retail_density"),
    round("distance", 3).alias("distance"),
    round("score", 3).alias("score")
)

# Write each as a single CSV folder
serving_df.coalesce(1).write.mode("overwrite").option("header", True).csv(
    PROCESSED_PATH + "snowflake_tract_master"
)

top_candidates_export_df.coalesce(1).write.mode("overwrite").option("header", True).csv(
    PROCESSED_PATH + "snowflake_top_candidates"
)

print("Export complete.")

Export complete.
